# ADC 2023 Atmospheric Retrieval with JAX

This notebook demonstrates how to use the JAX-based atmospheric retrieval pipeline on Ariel Data Challenge 2023 data.

**Features:**
- Pure JAX gradient-based optimization (no TensorFlow/Keras)
- GPU acceleration on RTX 3080 Ti
- Fully differentiable forward model
- Clean functional API (no class abstractions)

**Molecules fitted:** H2O, CO2, CO, CH4, NH3

## Setup

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from taurex.cache import OpacityCache, CIACache
from adc_jax_utils import (
    load_planet_spectrum,
    load_auxiliary_data,
    load_ground_truth,
    create_taurex_model_from_adc_planet,
    create_adc_jax_forward_model,
    fit_adc_planet,
    plot_adc_fit_results
)

## Configure Precision

Choose float64 (safer, ~1.5x slower) or float32 (~2x faster)

In [ ]:
# Set precision
USE_FLOAT64 = True  # Change to False for float32

jax.config.update("jax_enable_x64", USE_FLOAT64)
DTYPE = jnp.float64 if USE_FLOAT64 else jnp.float32

print(f"Precision: {'float64' if USE_FLOAT64 else 'float32'}")
print(f"Device: {jax.devices()[0]}")
print(f"Default dtype: {jnp.array(1.0).dtype}")

## Setup Opacity Cache

In [ ]:
# Setup paths
XSEC_PATH = 'test_files/xsec/xsec_sampled_R15000_0.3-50'
CIA_PATH = 'test_files/cia/HITRAN/data'

# Initialize caches
OpacityCache().clear_cache()
OpacityCache().set_opacity_path(XSEC_PATH)
CIACache().set_cia_path(CIA_PATH)

print("Opacity cache ready!")

## Load ADC Planet Data

Let's load a single planet from the training set.

In [ ]:
# Choose a planet
PLANET_ID = 1000  # Change to any planet ID (1-41423)

# Data paths
DATA_DIR = 'test_files/adc_2023/TrainingData'
SPECTRAL_PATH = f'{DATA_DIR}/SpectralData.hdf5'
AUX_PATH = f'{DATA_DIR}/AuxillaryTable.csv'
GT_PATH = f'{DATA_DIR}/Ground Truth Package/FM_Parameter_Table.csv'

# Load data
print(f"Loading planet {PLANET_ID}...")
spectrum_dict = load_planet_spectrum(SPECTRAL_PATH, PLANET_ID)
aux_data = load_auxiliary_data(AUX_PATH, PLANET_ID)
ground_truth = load_ground_truth(GT_PATH, PLANET_ID)

print(f"\nSpectrum info:")
print(f"  Wavelength range: {spectrum_dict['wl_grid'].min():.2f} - {spectrum_dict['wl_grid'].max():.2f} μm")
print(f"  Spectral bins: {len(spectrum_dict['spectrum'])}")

print(f"\nStellar parameters:")
print(f"  Temperature: {aux_data['star_temperature']:.1f} K")
print(f"  Radius: {aux_data['star_radius_m']/6.96e8:.2f} R_sun")

print(f"\nGround truth atmospheric parameters:")
print(f"  Planet radius: {ground_truth['planet_radius']:.3f} R_jup")
print(f"  Temperature: {ground_truth['planet_temp']:.1f} K")
print(f"  H2O: {ground_truth['H2O']:.3e}")
print(f"  CO2: {ground_truth['CO2']:.3e}")
print(f"  CO: {ground_truth['CO']:.3e}")
print(f"  CH4: {ground_truth['CH4']:.3e}")
print(f"  NH3: {ground_truth['NH3']:.3e}")

## Visualize Observed Spectrum

In [ ]:
plt.figure(figsize=(12, 4))
plt.errorbar(spectrum_dict['wl_grid'], spectrum_dict['spectrum'], 
             yerr=spectrum_dict['noise'], fmt='o', alpha=0.7, markersize=5)
plt.xlabel('Wavelength (μm)')
plt.ylabel('Transit Depth')
plt.title(f'ADC Planet {PLANET_ID} - Observed Spectrum')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Method 1: One-Line Fitting (Easiest)

Use the convenience function that handles everything.

In [ ]:
# Simple one-line fit
result = fit_adc_planet(
    planet_id=PLANET_ID,
    data_dir=DATA_DIR,
    fit_params=['planet_radius', 'T', 'H2O', 'CO2', 'CO', 'CH4', 'NH3'],
    steps=500,
    lr=1e-3,
    nlayers=30,
    dtype=DTYPE,
    verbose=True
)

In [ ]:
# Plot results
plot_adc_fit_results(result, save_path=f'adc_planet_{PLANET_ID}_results.png')

## Method 2: Step-by-Step (More Control)

For more control, build the pipeline manually.

In [ ]:
# Create TauREx model from ADC parameters
print("Creating TauREx model...")
tm = create_taurex_model_from_adc_planet(
    aux_data,
    ground_truth=ground_truth,  # Use ground truth for initialization
    nlayers=30,
    active_molecules=['H2O', 'CO2', 'CO', 'CH4', 'NH3']
)

print(f"  Layers: {tm.nLayers}")
print(f"  Active gases: {list(tm.chemistry.activeGases)}")
print(f"  Contributions: {[c.name for c in tm.contribution_list]}")

In [ ]:
# Create JAX differentiable forward model
print("Creating JAX forward model...")
forward_model, obs_spectrum = create_adc_jax_forward_model(
    tm,
    spectrum_dict,
    use_full_diff=True,  # Use fully differentiable version
    dtype=DTYPE
)

print("  Forward model ready!")

In [ ]:
# Extract parameters and run optimization
from experiment import extract_fitting_params, fit_with_value_and_grad_adam

params, param_info = extract_fitting_params(tm, dtype=DTYPE)
obs_y = jnp.asarray(spectrum_dict['spectrum'], dtype=DTYPE)
obs_err = jnp.asarray(spectrum_dict['noise'], dtype=DTYPE)

fit_params = ['planet_radius', 'T', 'H2O', 'CO2', 'CO', 'CH4', 'NH3']

print("Starting optimization...")
final_params, losses = fit_with_value_and_grad_adam(
    forward_binned=forward_model,
    observed_y=obs_y,
    observed_err=obs_err,
    init_params=params,
    param_info=param_info,
    fit_params=fit_params,
    steps=500,
    lr=1e-3,
    clip_norm=1.0,
    print_every=50,
    nan_guard=True,
    loss="mse"
)

In [ ]:
# Plot loss history
plt.figure(figsize=(10, 4))
plt.semilogy(losses)
plt.xlabel('Iteration')
plt.ylabel('Loss (MSE)')
plt.title('Optimization Loss History')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compare fitted vs ground truth
print("\nFinal Results:")
print("-" * 70)
print(f"{'Parameter':<15} {'Initial':<15} {'Fitted':<15} {'Ground Truth':<15} {'Error %':<10}")
print("-" * 70)

for param in fit_params:
    initial = float(params[param])
    final = float(final_params[param])
    gt = float(ground_truth[param])
    error = abs(final - gt) / abs(gt) * 100 if gt != 0 else abs(final - gt)
    
    print(f"{param:<15} {initial:<15.6e} {final:<15.6e} {gt:<15.6e} {error:<10.2f}")

## Visualize Fit

In [ ]:
# Compute spectra
initial_spectrum = np.array(forward_model(params))
final_spectrum = np.array(forward_model(final_params))

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Spectra comparison
ax = axes[0]
ax.errorbar(spectrum_dict['wl_grid'], spectrum_dict['spectrum'], 
            yerr=spectrum_dict['noise'], fmt='o', alpha=0.5, 
            label='Observed', markersize=4)
ax.plot(spectrum_dict['wl_grid'], initial_spectrum, '--', 
        label='Initial model', linewidth=2)
ax.plot(spectrum_dict['wl_grid'], final_spectrum, '-', 
        label='Fitted model', linewidth=2)
ax.set_xlabel('Wavelength (μm)')
ax.set_ylabel('Transit Depth')
ax.set_title(f'ADC Planet {PLANET_ID} - Atmospheric Retrieval')
ax.legend()
ax.grid(True, alpha=0.3)

# Residuals
ax = axes[1]
initial_residuals = (spectrum_dict['spectrum'] - initial_spectrum) / spectrum_dict['noise']
final_residuals = (spectrum_dict['spectrum'] - final_spectrum) / spectrum_dict['noise']

ax.axhline(0, color='black', linestyle='--', alpha=0.5)
ax.scatter(spectrum_dict['wl_grid'], initial_residuals, alpha=0.5, label='Initial residuals')
ax.scatter(spectrum_dict['wl_grid'], final_residuals, alpha=0.5, label='Final residuals')
ax.set_xlabel('Wavelength (μm)')
ax.set_ylabel('Residuals (σ)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal χ²: {np.sum(final_residuals**2):.2f}")
print(f"Reduced χ²: {np.sum(final_residuals**2) / len(final_residuals):.2f}")

## Try Different Planets

Change `PLANET_ID` and re-run to test on different planets!

In [ ]:
# Quick test on multiple planets
test_planets = [100, 500, 1000, 2000, 5000]

for pid in test_planets:
    print(f"\nFitting planet {pid}...")
    try:
        result = fit_adc_planet(
            planet_id=pid,
            data_dir=DATA_DIR,
            fit_params=['planet_radius', 'T', 'H2O'],  # Fit fewer params for speed
            steps=200,
            lr=1e-3,
            nlayers=20,  # Fewer layers for speed
            dtype=DTYPE,
            verbose=False
        )
        print(f"  Final loss: {result['losses'][-1]:.6e}")
        
        if result['ground_truth'] is not None:
            for param in ['planet_radius', 'T', 'H2O']:
                final = float(result['final_params'][param])
                gt = float(result['ground_truth'][param])
                error = abs(final - gt) / abs(gt) * 100 if gt != 0 else 0
                print(f"  {param}: {error:.1f}% error")
    except Exception as e:
        print(f"  Error: {e}")

## Summary

This notebook demonstrated:
1. Loading ADC 2023 planet data (spectrum, auxiliary info, ground truth)
2. Creating a TauREx transmission model from planet parameters
3. Creating a JAX differentiable forward model
4. Running gradient-based optimization with Adam
5. Comparing results to ground truth

**Next steps:**
- Try different planets
- Experiment with learning rates and optimization steps
- Test float32 vs float64 performance
- Fit more/fewer parameters
- Adjust number of atmospheric layers